# Build the verified recursion-to-iteration dataset on a T4

Same script, same gate, same inputs as the local run. Only the generation
backend changes: `--backend hf` samples a whole function's attempts in one
batch on the GPU instead of one call at a time on the CPU.

| | CPU (llama-server) | T4 (batched) |
|---|---:|---:|
| per function, 6 samples | ~120 s | a few seconds |
| 582 functions | ~19 hours | ~1-2 hours |

**The GPU does not improve the yield.** Same weights, same answers. It makes
attempts cheap enough to afford more of them, and the failures are correlated -
a function the model will not de-recurse tends to stay that way - so more
samples help sublinearly. Plan on 15-25%, not 80%.

Once generation is fast, **compiling and running the candidates becomes the
bottleneck**, and that is CPU work Colab gives you two cores for. That is the
real limit on this notebook, not the model.

Runtime → Change runtime type → **T4 GPU** before running anything.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!g++ --version | head -1   # the gate needs this; Colab already has it

## 1. The code

Cloned rather than pasted, so the gate that decides which rows exist is the
same one with tests behind it. A private repo needs a token with `repo` scope.

In [ ]:
import os
from getpass import getpass

REPO = "safi892/fyp_training"
BRANCH = "language"

token = getpass("GitHub token (blank if the repo is public): ").strip()
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

!rm -rf /content/fyp && git clone -q -b {BRANCH} {url} /content/fyp
os.chdir("/content/fyp")
!git log --oneline -1

In [ ]:
# Kept narrow on purpose: Colab's preinstalled torch is fine and reinstalling it
# costs several minutes and sometimes a restart.
!pip install -q transformers peft accelerate
!pip list 2>/dev/null | grep -E "^(torch|transformers|peft|accelerate) "

## 2. The corpus

`cleaned/merged_cleaned.jsonl` is git-ignored, so it has to arrive another way.
Drive is the least painful; the file is around 40 MB.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Upload these to Drive first:
#   MyDrive/fyp/merged_cleaned.jsonl        the corpus
#   MyDrive/fyp/last_adapter/               adapter_config.json + adapter_model.safetensors
#                                           + tokenizer files (295 MB; optimizer.pt is NOT needed)
CORPUS = "/content/drive/MyDrive/fyp/merged_cleaned.jsonl"
ADAPTER = "/content/drive/MyDrive/fyp/last_adapter"

!ls -la {CORPUS} && ls {ADAPTER}

## 3. Check the inputs before spending GPU time

This should print the same counts as the laptop. If it does not, the corpus
that arrived is not the corpus the numbers were measured on, and nothing below
is comparable.

In [ ]:
import sys

sys.path.insert(0, "/content/fyp/src")
sys.path.insert(0, "/content/fyp/scripts")
from pathlib import Path

from build_optimize_dataset import WORDING, drivable_recursive, judge

functions = drivable_recursive(Path(CORPUS), 40)
print(f"{len(functions)} drivable recursive functions   (expected 582)")
print(f"wording: {WORDING[:60]}...")

# The gate itself, on a case whose answer is known. A wrong loop must be
# rejected here or every number this notebook produces is worthless.
rec = "int fact(int n){ if(n<=1) return 1; return n*fact(n-1); }"
itr = "int fact(int n){ int r=1; for(int i=2;i<=n;i++) r*=i; return r; }"
assert judge(rec, itr, 10.0) is None
assert judge(rec, itr.replace("r=1", "r=0"), 10.0) == "different output"
assert judge(rec, rec, 10.0) == "still recursive"
print("gate rejects a wrong rewrite and a non-rewrite: ok")

## 4. A short run first

Twenty functions, to get a real yield and a real per-function time before
committing the session. Sixteen samples rather than six, because on a GPU they
cost about the same as one.

In [ ]:
OUT = "/content/drive/MyDrive/fyp/verified.jsonl"   # on Drive, so a dropped session keeps it

!cd /content/fyp && python scripts/build_optimize_dataset.py \
    --backend hf --base Qwen/Qwen2.5-Coder-1.5B-Instruct --adapter {ADAPTER} \
    --corpus {CORPUS} --out {OUT} \
    --limit 20 --samples 16 --temperature 0.9

**Read the yield before going on.** Rows kept ÷ 20 is what 582 will give you.
At 20% that is about 115 rows; at 8% it is 47 and the GPU has bought speed
rather than a dataset. Either is a result worth writing down - the second one
says the limit is the model and not the budget.

## 5. The rest

Resumable: it skips functions already kept and already failed, so a disconnect
costs only the function in flight. Re-run this cell after a reconnect.

In [ ]:
!cd /content/fyp && python scripts/build_optimize_dataset.py \
    --backend hf --base Qwen/Qwen2.5-Coder-1.5B-Instruct --adapter {ADAPTER} \
    --corpus {CORPUS} --out {OUT} \
    --limit 582 --samples 16 --temperature 0.9

## 6. What came out

Re-verified here rather than trusted, because the rows were written by a
process that could have been interrupted mid-line.

In [ ]:
import json

rows = [json.loads(line) for line in open(OUT) if line.strip()]
print(f"{len(rows)} verified rows")

bad = [r for r in rows if judge(r["code"], r["improved_code"], 10.0) is not None]
print(f"{len(bad)} fail a second check   (must be 0)")

if rows:
    print("\n--- one row ---")
    print(rows[0]["code"][:300])
    print("    ->")
    print(rows[0]["improved_code"][:300])

Then download `verified.jsonl` from Drive into
`my_data_annotation/recursion_optimization/`. The local CPU run writes to the
same file in the same format, so the two merge by concatenation - de-duplicate
on `code`, since both runs draw from the same 582 inputs.